# Quantum CQ - Simple API Lab

Notebook oficial de validacao manual da API simples da RUN 3.7.

In [ ]:
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

warnings.filterwarnings("default")
%matplotlib inline

from quantum_cq import CQ

RUN_IDEAL = True
RUN_NOISY_LOCAL = False
RUN_REAL_IBM = False

In [ ]:
def section(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def safe_show(obj, title=None):
    if title:
        section(title)
    try:
        return CQ.show(obj)
    except Exception as exc:
        print("CQ.show falhou; usando CQ.describe:", type(exc).__name__, exc)
        CQ.describe(obj)
        return obj

def safe_run(label, fn):
    section(label)
    try:
        return fn()
    except Exception as exc:
        print("Falhou:", type(exc).__name__, exc)
        return None

In [ ]:
required = [
    "state", "nav", "addressed", "graph_nav", "walk", "deutsch",
    "bv", "dj", "grover", "qpe", "qft", "iqft", "diffuser",
    "phase_rotation", "show", "draw", "describe", "run",
]
for name in required:
    assert hasattr(CQ, name), name
print("OK: facade simples disponivel")

In [ ]:
section("State encoders simples")
state_basis = CQ.state([1, 0, 1])
safe_show(state_basis, "CQ.state basis")
state_angle = CQ.state([0.1, 0.2, 0.3], encoding="angle")
safe_show(state_angle, "CQ.state angle")
state_phase = CQ.state([0.1, 0.2], encoding="phase")
safe_show(state_phase, "CQ.state phase")
state_amp = CQ.state([1, 0, 0, 0], encoding="amplitude")
safe_show(state_amp, "CQ.state amplitude")

In [ ]:
section("Addressed memory simples")
nav = CQ.nav([3, 5, 7, 9])
safe_show(nav, "CQ.nav explicit")
nav_sparse = CQ.nav([3, 5, 7, 9], engine="sparse")
CQ.describe(nav_sparse)
nav_qram = CQ.nav([3, 5, 7, 9], engine="qram")
CQ.describe(nav_qram)
addressed = CQ.addressed([3, 5, 7, 9])
CQ.describe(addressed)

In [ ]:
section("Graph navigation e walk")
graph = CQ.graph([(0, 1), (1, 2), (2, 3)], vertices=4)
graph_nav = CQ.graph_nav(graph)
safe_show(graph_nav, "CQ.graph_nav")
walk = CQ.walk(graph, steps=1)
safe_show(walk, "CQ.walk")

In [ ]:
section("Algoritmos simples")
deutsch = CQ.deutsch(case=2)
safe_show(deutsch, "CQ.deutsch")
bv = CQ.bv("1011")
safe_show(bv, "CQ.bv")
dj = CQ.dj(kind="balanced", qubits=3)
safe_show(dj, "CQ.dj")
grover = CQ.grover("11")
safe_show(grover, "CQ.grover")
qpe = CQ.qpe(phase=0.25, precision=3)
safe_show(qpe, "CQ.qpe")

In [ ]:
section("Operadores e primitivas simples")
qft = CQ.qft(3)
safe_show(qft, "CQ.qft")
iqft = CQ.iqft(3)
safe_show(iqft, "CQ.iqft")
diffuser = CQ.diffuser(3)
safe_show(diffuser, "CQ.diffuser")
rot = CQ.phase_rotation(0.25)
safe_show(rot, "CQ.phase_rotation")

In [ ]:
section("Pipeline simplificada")
p1 = CQ.pipeline([1, 0.3, 1]).build()
safe_show(p1, "CQ.pipeline(data_posicional)")
p2 = CQ.pipeline(data=[1, 0.3, 1]).build()
safe_show(p2, "CQ.pipeline(data=...)")
p_old = CQ.pipeline().with_data([1, 0.3, 1]).build()
safe_show(p_old, "API antiga preservada")

In [ ]:
section("CQ.run ideal")
if RUN_IDEAL:
    result = CQ.run(CQ.deutsch(case=2), mode="ideal", shots=128, title="Deutsch ideal via CQ.run")
    result.show()
    print(result.counts_for("ideal"))
    print(result.classify("deutsch"))

    result_multi = CQ.run(
        circuits=[CQ.deutsch(case=1), CQ.deutsch(case=2), CQ.bv("1011")],
        modes=["ideal"],
        shots=128,
        title="Multiplos circuitos ideal",
    )
    result_multi.show()
    print(result_multi.global_metrics())
    display(result_multi.to_dataframe())

In [ ]:
section("CQ.run com data/encoders")
if RUN_IDEAL:
    result_data = CQ.run(
        data=[0.1, 0.2, 0.3],
        encoders=["angle", "phase"],
        mode="ideal",
        shots=128,
        title="Dataset com multiplos encoders",
    )
    result_data.show()
    print(result_data.by_encoder())
    display(result_data.to_dataframe())

In [ ]:
section("IBM config segura sem execucao real")
ibm = CQ.ibm(token="COLE_SEU_TOKEN_AQUI", channel="ibm_quantum_platform", instance="")
print(ibm)
print(ibm.safe_summary())
assert "COLE_SEU_TOKEN_AQUI" not in repr(ibm)
assert "COLE_SEU_TOKEN_AQUI" not in str(ibm.safe_summary())

In [ ]:
section("API simples vs API avancada")
nav = CQ.nav([3, 5, 7, 9])
memory = CQ.memory([3, 5, 7, 9])
nav_old = CQ.encode(memory, role="navigation", engine="explicit_circuit")
print(nav.metadata["engine"], nav_old.metadata["engine"])

deutsch = CQ.deutsch(case=2)
deutsch_old = CQ.algorithm("deutsch").with_case(2).build()
print(deutsch.algorithm_name, deutsch_old.algorithm_name)

safe_run("BenchmarkingPipeline avancada", lambda: __import__("quantum_cq.pipeline"))

In [ ]:
print("OK: API simples carregada.")
print("OK: Encoders simples.")
print("OK: Navigation simples.")
print("OK: Graph/walk.")
print("OK: Algorithms simples.")
print("OK: Operators simples.")
print("OK: CQ.run ideal.")
print("OK: PipelineResult.")
print("OK: IBM config segura.")
print("OK: API avancada preservada.")
print("OK: quantum_cq_simple_api_lab.ipynb executou com sucesso.")